In [ ]:
import functools
import jax
import os

from datetime import datetime
from jax import numpy as jp
import matplotlib.pyplot as plt

from IPython.display import HTML, clear_output

import brax
import flax
from brax import envs
from brax.io import model
from brax.io import json
from brax.io import html
# from brax.training.agents.ppo import train as ppo
# from brax.training.agents.sac import train as sac

import functools
import time
from typing import Any, Callable, Mapping, Optional, Tuple, Union

from absl import logging
from brax import base
from brax import envs
from brax.training import acting
from brax.training import gradients
from brax.training import pmap
from brax.training import types
from brax.training.acme import running_statistics
from brax.training.acme import specs
from brax.training.agents.ppo import losses as ppo_losses
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.types import Params
from brax.training.types import PRNGKey
from brax.v1 import envs as envs_v1
from etils import epath
import flax
import jax
import jax.numpy as jnp
import numpy as np
import optax
from orbax import checkpoint as ocp

from brax.envs.base import PipelineEnv, State
from brax.io import mjcf
from etils import epath

import wandb
import xmltodict

from ppo import ppo_train
from env import HalfcheetahWithObstacles

os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

run = wandb.init(
    project='test',
    group='vu',
    name='zuxinrui',
    mode="online",
)

wandb: Currently logged in as: zuxinrui to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [13]:
env = envs.get_environment(env_name='pusher', backend='mjx')
state = jax.jit(env.reset)(rng=jax.random.PRNGKey(seed=0))
url = html.render(env.sys.tree_replace({"opt.timestep": env.dt}), [state.pipeline_state], height=1024)
wandb.log({"env render": wandb.Html(url)})

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x763be7dcc8e0>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 76383d21db40, raw_cell="env = envs.get_environment(env_name='pusher', back.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/zuxinrui/JaxGCRL/notebooks/brax_wandb_before_ALife.ipynb#X10sZmlsZQ%3D%3D>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

BrokenPipeError: [Errno 32] Broken pipe

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x763be7dcc8e0>> (for post_run_cell), with arguments args (<ExecutionResult object at 76383d21db10, execution_count=13 error_before_exec=None error_in_exec=[Errno 32] Broken pipe info=<ExecutionInfo object at 76383d21db40, raw_cell="env = envs.get_environment(env_name='pusher', back.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/zuxinrui/JaxGCRL/notebooks/brax_wandb_before_ALife.ipynb#X10sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

In [11]:
env = HalfcheetahWithObstacles(
    obstacle_height=0.4,  # (0.2 - 0.5)
    obstacle_width=0.2,  # (0.1 - 0.5)
    obstacle_spacing=1.0,  # (0.5 - 2.0)
    n_obstacles=10,  # 10
    design=None,
    backend='mjx',
)
state = jax.jit(env.reset)(rng=jax.random.PRNGKey(seed=0))

url = html.render(env.sys.tree_replace({"opt.timestep": env.dt}), [state.pipeline_state], height=1024)
# with open(os.path.join(exp_dir, f"{exp_name}_{num_steps}.html"), "w") as file:
#     file.write(url)
wandb.log({"env render": wandb.Html(url)})

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x763be7dcc8e0>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 763a38710340, raw_cell="env = HalfcheetahWithObstacles(
    obstacle_heigh.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/zuxinrui/JaxGCRL/notebooks/brax_wandb_before_ALife.ipynb#X11sZmlsZQ%3D%3D>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

Changing morphology 1.2908039176951063 1.7065131202635007 0.5738654872568562 0.6746791176559019 0.9744667869534737 1.38839472512104


BrokenPipeError: [Errno 32] Broken pipe

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x763be7dcc8e0>> (for post_run_cell), with arguments args (<ExecutionResult object at 763a381dc700, execution_count=11 error_before_exec=None error_in_exec=[Errno 32] Broken pipe info=<ExecutionInfo object at 763a38710340, raw_cell="env = HalfcheetahWithObstacles(
    obstacle_heigh.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/zuxinrui/JaxGCRL/notebooks/brax_wandb_before_ALife.ipynb#X11sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

In [ ]:
episode_length = 1000

# # for halfcheetah:
# train_fn = functools.partial(
#     ppo_train,
#     num_timesteps=10_000_000,
#     num_evals=3,
#     reward_scaling=1,
#     episode_length=episode_length,
#     normalize_observations=True,
#     action_repeat=1,
#     unroll_length=20,
#     num_minibatches=32,
#     num_updates_per_batch=8,
#     discounting=0.95,
#     learning_rate=3e-4,
#     entropy_cost=0.001,
#     num_envs=4096,  # 2048 on 4070 ti s is the fastest  p.s.: num_envs must be divisible by n_batch * batch_size (1+ times per env simulation in the batch)
#     batch_size=128,
#     seed=3,
# )

# for pusher:
train_fn = functools.partial(
    ppo_train,
    num_timesteps=100_000_000,
    num_evals=10,
    reward_scaling=5,
    episode_length=episode_length,  # 1000
    normalize_observations=True,
    action_repeat=1,
    unroll_length=30,
    num_minibatches=16,
    num_updates_per_batch=8,
    discounting=0.95,
    learning_rate=3e-4,
    entropy_cost=1e-2,
    num_envs=2048,  # 2048 on 4070 ti s is the fastest  p.s.: num_envs must be divisible by n_batch * batch_size (1+ times per env simulation in the batch)
    batch_size=512,
    seed=3,
)

xdata, ydata = [], []
times = [datetime.now()]

def progress(num_steps, metrics, params, make_policy):
    render(make_policy, params, env, './logs/htmls/', 'halfcheetah', num_steps, metrics)
    times.append(datetime.now())

def render(make_policy, params, env, exp_dir, exp_name, num_steps, metrics=None):
    policy = make_policy(params)
    jit_env_reset = jax.jit(env.reset)
    jit_env_step = jax.jit(env.step)
    jit_policy = jax.jit(policy)

    rollout = []
    key = jax.random.PRNGKey(seed=1)
    key, subkey = jax.random.split(key)
    state = jit_env_reset(rng=subkey)
    for i in range(episode_length):  # 1000 = 50s
        rollout.append(state.pipeline_state)
        key, subkey = jax.random.split(key)
        action, _ = jit_policy(state.obs, subkey)  # Policy requires batched dimension
        # action = action[0]  # Remove batch dimension
        state = jit_env_step(state, action)
        # if i % 1000 == 0:
        #     key, subkey = jax.random.split(key)
        #     state = jit_env_reset(rng=subkey)

    url = html.render(env.sys.tree_replace({"opt.timestep": env.dt}), rollout, height=1024)
    with open(os.path.join(exp_dir, f"{exp_name}_{num_steps}.html"), "w") as file:
        file.write(url)
    wandb.log({
        "video": wandb.Html(url),
        'training/reward': metrics['eval/episode_reward'],
    })

make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)

print(f'time to jit: {times[1] - times[0]}')
print(f'time to train: {times[-1] - times[1]}')
print(f'time overall: {times[-1] - times[0]}')
# n minibatch really doesn't matter too much!

In [7]:
inference_fn = make_inference_fn(params)
jit_env_reset = jax.jit(env.reset)
jit_env_step = jax.jit(env.step)
jit_inference_fn = jax.jit(inference_fn)

rollout = []
rng = jax.random.PRNGKey(seed=1)
state = jit_env_reset(rng=rng)
for _ in range(200):
  rollout.append(state.pipeline_state)
  act_rng, rng = jax.random.split(rng)
  act, _ = jit_inference_fn(state.obs, act_rng)
  state = jit_env_step(state, act)

url = html.render(env.sys.tree_replace({'opt.timestep': env.dt}), rollout)
wandb.log({"video": wandb.Html(url)})

In [22]:
run.finish()

Error in callback <bound method _WandbInit._pre_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x763be7dcc8e0>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 763ac56daad0, raw_cell="run.finish()" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/zuxinrui/JaxGCRL/notebooks/brax_wandb_before_ALife.ipynb#W5sZmlsZQ%3D%3D>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

BrokenPipeError: [Errno 32] Broken pipe

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x763be7dcc8e0>> (for post_run_cell), with arguments args (<ExecutionResult object at 763ac56dab60, execution_count=22 error_before_exec=None error_in_exec=[Errno 32] Broken pipe info=<ExecutionInfo object at 763ac56daad0, raw_cell="run.finish()" store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/home/zuxinrui/JaxGCRL/notebooks/brax_wandb_before_ALife.ipynb#W5sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe